# 01 — Data Exploration

Explore ModelNet40 and ShapeNet datasets used for training GeoFusion.

**Topics:**
- Loading and inspecting 3D point clouds
- Visualization of shapes and classes
- Data augmentation pipeline
- Synthetic text metadata generation

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from geofusion.data.datasets import ModelNet40Dataset
from geofusion.data.transforms import (
    Compose, NormalizePointCloud, FarthestPointSample,
    RandomRotate, RandomJitter, RandomScale
)
from geofusion.data.text_metadata import TextMetadataGenerator

## Load ModelNet40

In [ ]:
transform = Compose([
    FarthestPointSample(2048),
    NormalizePointCloud(),
])

# Update path to your downloaded data
DATA_ROOT = '../data/raw/modelnet40_normal_resampled'

try:
    dataset = ModelNet40Dataset(data_root=DATA_ROOT, split='train', transform=transform)
    print(f'Dataset size: {len(dataset)}')
    print(f'Classes ({len(dataset.classes)}): {dataset.classes[:10]}...')
except FileNotFoundError:
    print('Dataset not found. Run: python scripts/download_data.py')
    dataset = None

## Visualize Point Clouds

In [ ]:
def plot_point_cloud(points, title='Point Cloud', ax=None):
    """Visualize a 3D point cloud."""
    if ax is None:
        fig = plt.figure(figsize=(8, 8))
        ax = fig.add_subplot(111, projection='3d')
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], s=1, alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    return ax

if dataset is not None:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12),
                             subplot_kw={'projection': '3d'})
    for i, ax in enumerate(axes.flat):
        idx = np.random.randint(len(dataset))
        sample = dataset[idx]
        pts = sample['points'].numpy() if hasattr(sample['points'], 'numpy') else sample['points']
        plot_point_cloud(pts[:, :3], title=sample['class_name'], ax=ax)
    plt.tight_layout()
    plt.show()

## Data Augmentation Pipeline

In [ ]:
if dataset is not None:
    sample = dataset[0]
    original = sample['points'].numpy()[:, :3]

    augmentations = {
        'Original': lambda x: x,
        'Rotated': RandomRotate(axis='y'),
        'Jittered': RandomJitter(sigma=0.02),
        'Scaled': RandomScale(lo=0.7, hi=1.3),
    }

    fig, axes = plt.subplots(1, 4, figsize=(24, 6),
                             subplot_kw={'projection': '3d'})
    for ax, (name, aug) in zip(axes, augmentations.items()):
        pts = aug(original.copy())
        plot_point_cloud(pts, title=name, ax=ax)
    plt.tight_layout()
    plt.show()

## Synthetic Text Metadata

In [ ]:
text_gen = TextMetadataGenerator(seed=42)

categories = ['airplane', 'chair', 'car', 'table', 'lamp']
for cat in categories:
    desc = text_gen.generate(cat)
    print(f'[{cat:>10s}]  {desc}\n')